### Topics in NLP

1. Loading Data
2. EDA
3. Data Preprocessing
   3.1 Cleaning the corpus
   3.2 Stemming
   3.3 All Together
   3.4 Target Encoding
4. Tokens Visualization
5. Vectorization
   5.1 Tunning CountVectorizer
   5.2 TF-IDF
   5.3 Word Embeddings: Glove
6. Modeling
   6.1 Naive Bayes DTM
   6.2 Naive Bayes TF-IDF
   6.3 XGBoost
7. LSTM
8. BERT
9. NLP: Disaster Tweets
   9.1 EDA


Steps of NLP
1. Text Collection
2. Text Preprocessing
    a. Converting everything to lowercase
    b. Removing punctuation(commas, periods, exclamation marks, etc.)
    c. Removing stop words (common words like "the", "is", "in", etc.)
    d. Tokenization (splitting text into individual words or tokens)
    e. Stemming or Lemmatization (reducing words to their base or root form)
3. Feature Extraction
    a. Vectorization (converting tokens into vectors)
    OR
    b. Embeddings(Embedding/Mapping the individual words to its numerical representation)
4. Modeling

In [69]:
import re
import string
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
from plotly import graph_objs as go
import plotly.express as px
import plotly.figure_factory as ff
from collections import Counter
import nltk
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from PIL import Image
nltk.download('stopwords') ## Downloading Stopword corpus
nltk.download('punkt') ## Downloading Tokenization
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import os

import spacy
import random
from spacy.util import compounding
from spacy.util import minibatch
from collections import defaultdict
from collections import Counter
import keras
from keras.models import Sequential
from keras.initializers import Constant
from keras.layers import (LSTM,
                          Embedding,
                          BatchNormalization,
                          Dense,
                          TimeDistributed,
                          Dropout,
                          Bidirectional,
                          Flatten,
                          GlobalMaxPool1D)
# from keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.preprocessing.sequence import pad_sequences
from keras.layers import Embedding
# from keras.layers.embeddings import Embedding
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    accuracy_score
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\redab\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\redab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\redab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [70]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [71]:
primary_blue="#496595"
primary_blue2="#85a1c1"
primary_blue3="#3f4d63"
primary_grey="#c6ccd8"
primary_black="#202022"
primary_bgcolor="f4f0ea"

primary_green=px.colors.qualitative.Plotly[2]

In [72]:
df=pd.read_csv("datasets\spam.csv",encoding="latin-1")

In [73]:
df.describe()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
count,5572,5572,50,12,6
unique,2,5169,43,10,5
top,ham,"Sorry, I'll call later","bt not his girlfrnd... G o o d n i g h t . . .@""","MK17 92H. 450Ppw 16""","GNT:-)"""
freq,4825,30,3,2,2


In [74]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [75]:
df=df.dropna(how="any",axis=1)
df.columns=["target","message"]

In [76]:
df.head()

,target,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [77]:
df['message_len']=df['message'].apply(lambda x: len(x.split(' ')))
df.head()

,target,message,message_len
0,ham,"Go until jurong point, crazy.. Available only ...",20
1,ham,Ok lar... Joking wif u oni...,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28
3,ham,U dun say so early hor... U c already then say...,11
4,ham,"Nah I don't think he goes to usf, he lives aro...",13


In [78]:
print("Max Length:",max(df['message_len']))
print("Min Length:",min(df['message_len']))

Max Length: 171
Min Length: 1


In [79]:
df['message'].apply(lambda x: len(x.split('. ')))

0       3
1       2
2       2
3       2
4       1
       ..
5567    4
5568    1
5569    2
5570    1
5571    2
Name: message, Length: 5572, dtype: int64

In [80]:
df['target'].value_counts()

target
ham     4825
spam     747
Name: count, dtype: int64

In [81]:
balance_counts=pd.DataFrame({'target': df['target'].value_counts().index, 'count': df['target'].value_counts().values})

In [82]:
balance_counts

,target,count
0,ham,4825
1,spam,747


In [83]:
print(balance_counts['count'][0])

4825


In [84]:
fig=go.Figure()

fig.add_trace(go.Bar(
    x=['ham'],
    y=[balance_counts['count'][0]],
    name='ham',
    text=[balance_counts['count'][0]],
    marker_color=primary_blue
))

fig.add_trace(go.Bar(
    x=['spam'],
    y=[balance_counts['count'][1]],
    name='spam',
    text=[balance_counts['count'][1]],
    marker_color=primary_green
))

fig.update_layout(
    title="<span style='font-size: 20px;'>Spam vs Ham Count</span>",
    xaxis_title="<span style='font-size: 16px;'>Target</span>",
    yaxis_title="<span style='font-size: 16px;'>Count</span>",
    plot_bgcolor='#f4f0ea',   # Fixed: added '#' prefix
    paper_bgcolor='#f4f0ea',  # Make sure this has '#' too if you used it
)

fig.show()

In [85]:
ham_df=df[df['target']=='ham']['message_len'].value_counts().sort_index()
spam_df=df[df['target']=='spam']['message_len'].value_counts().sort_index()

fig=go.Figure()

fig.add_trace(go.Scatter(
    x=ham_df.index,
    y=ham_df.values,
    name='ham',
    fill='tozeroy',
    marker_color=primary_blue
))

fig.add_trace(go.Scatter(
    x=spam_df.index,
    y=spam_df.values,
    name='spam',
    fill='tozeroy',
    marker_color=primary_grey
))

fig.update_layout(
    title="<span style='font-size: 20px;'>Spam vs Ham Count</span>",
    xaxis_title="<span style='font-size: 16px;'>Target</span>",
    yaxis_title="<span style='font-size: 16px;'>Count</span>",
    plot_bgcolor='#f4f0ea',   # Fixed: added '#' prefix
    paper_bgcolor='#f4f0ea',  # Make sure this has '#' too if you used it
)

fig.update_xaxes(range=[0, 100])
fig.update_yaxes(range=[0, max(ham_df.max(), spam_df.max())])

fig.show()

In [86]:
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [87]:
df['message_clean']=df['message'].apply(clean_text)
df.head()

,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entry in a wkly comp to win fa cup final...
3,ham,U dun say so early hor... U c already then say...,11,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah i dont think he goes to usf he lives aroun...


In [88]:
%%capture
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\redab\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [89]:

stop_words=stopwords.words('english')
more_stopwords=['u','im','c']
stop_words=stop_words+more_stopwords

In [90]:
def remove_stopwords(text):
    text=' '.join(word for word in text.split(' ') if word not in stop_words)
    return text

In [91]:
df['message_clean']=df['message_clean'].apply(remove_stopwords)
df.head()

,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joking wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entry wkly comp win fa cup final tkts m...
3,ham,U dun say so early hor... U c already then say...,11,dun say early hor already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah dont think goes usf lives around though


In [92]:
stemmer=nltk.SnowballStemmer("english")

def steam_text(text):
    text=' '.join(stemmer.stem(word) for word in text.split(' '))
    return text

In [93]:
df['message_clean']=df['message_clean'].apply(steam_text)
df.head()

,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entri wkli comp win fa cup final tkts m...
3,ham,U dun say so early hor... U c already then say...,11,dun say earli hor alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah dont think goe usf live around though


In [94]:
def preprocess_data(text):
    # Text Cleaning
    text=clean_text(text)
    # Remove StopWords
    text=' '.join(word for word in text.split(' ') if word not in stop_words)
    # Stemming
    text=' '.join(stemmer.stem(word) for word in text.split(' '))

    return text

In [95]:
df['message_clean']=df['message_clean'].apply(preprocess_data)
df.head()

,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entri wkli comp win fa cup final tkts m...
3,ham,U dun say so early hor... U c already then say...,11,dun say ear hor alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah dont think goe usf live around though


In [96]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
le.fit(df['target'])

df['target_encoded']=le.transform(df['target'])
df.head(4)

,target,message,message_len,message_clean,target_encoded
0,ham,"Go until jurong point, crazy.. Available only ...",20,go jurong point crazi avail bugi n great world...,0
1,ham,Ok lar... Joking wif u oni...,6,ok lar joke wif oni,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entri wkli comp win fa cup final tkts m...,1
3,ham,U dun say so early hor... U c already then say...,11,dun say ear hor alreadi say,0


In [ ]:
texts=df['message_clean']
targets=df['target_encoded']

array([[   2, 3179,  274, ...,    0,    0,    0],
       [   8,  236,  527, ...,    0,    0,    0],
       [   9,  356,  588, ...,    0,    0,    0],
       ...,
       [6724, 1002, 6725, ...,    0,    0,    0],
       [ 138, 1251, 1603, ...,    0,    0,    0],
       [1986,  378,  170, ...,    0,    0,    0]],
      shape=(5572, 80), dtype=int32)

In [ ]:
word_tokenizer=Tokenizer()
word_tokenizer.fit_on_texts(texts)

vocab_length=len(word_tokenizer.word_index)+1
vocab_length

In [100]:
def embed(corpus):
    return word_tokenizer.texts_to_sequences(corpus)

longest_train=max(texts,key=lambda sentence: len(word_tokenize(sentence)))
length_long_sentence=len(word_tokenize(longest_train))

train_padded_sequences=pad_sequences(
    embed(texts),
    length_long_sentence,
    padding='post'
)

train_padded_sequences

array([[   2, 3179,  274, ...,    0,    0,    0],
       [   8,  236,  527, ...,    0,    0,    0],
       [   9,  356,  588, ...,    0,    0,    0],
       ...,
       [6724, 1002, 6725, ...,    0,    0,    0],
       [ 138, 1251, 1603, ...,    0,    0,    0],
       [1986,  378,  170, ...,    0,    0,    0]],
      shape=(5572, 80), dtype=int32)

Glove

Glove (Global Vectors for Word Representation) is an unsupervised learning algorithm for obtaining vector representations for words. It is based on the idea that the meaning of a word can be inferred from the context in which it appears. GloVe constructs a co-occurrence matrix of words in a corpus and then factorizes this matrix to obtain word vectors.

In [101]:
embedding_dictionary=dict()
embedding_dim=100

with open('datasets/glove.6B.100d.txt', encoding='utf-8') as fp:
    for line in fp.readlines():
        records=line.split()
        word=records[0]
        vector_dimensions=np.asarray(records[1:], dtype='float32')
        embedding_dictionary[word]=vector_dimensions

In [ ]:
len(embedding_dictionary)

400000


array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.57832998, -0.0036551 ,  0.34658   , ...,  0.070204  ,
         0.44509   ,  0.24147999],
       [-0.078894  ,  0.46160001,  0.57779002, ...,  0.26352   ,
         0.59397   ,  0.26741001],
       ...,
       [ 0.63009   , -0.036992  ,  0.24052   , ...,  0.10029   ,
         0.056822  ,  0.25018999],
       [-0.12002   , -1.23870003, -0.23303001, ...,  0.13658001,
        -0.61848003,  0.049843  ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(6726, 100))

In [ ]:

embedding_matrix=np.zeros((vocab_length,embedding_dim))

for word, index in word_tokenizer.word_index.items():
    embedding_vector=embedding_dictionary.get(word)
    if embedding_vector is not None:
        embedding_matrix[index]=embedding_vector

embedding_matrix

In [ ]:
from sklearn.naive_bayes import MultinomialNB
nb=MultinomialNB()

# nb.fit(x_train_dtm,y_train)